In [76]:
import os
import torch
import numpy as np
import cv2

from PIL import Image
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F   # 🔥 ADD THIS

from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = "/Users/arnavkarnik/Desktop/AIBD_Project/Aishwarya thesis"

CLASSES = ["PV", "PF", "BP", "EBA", "Vasculitis"]
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20

Using device: cpu


In [77]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # 🔥 ADD THIS
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [78]:
class AIBDDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for cls in CLASSES:
            cls_path = os.path.join(root_dir, cls)
            if not os.path.exists(cls_path):
                continue

            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                    self.samples.append((
                        os.path.join(cls_path, img_name),
                        CLASS_TO_IDX[cls]
                    ))

        print("Total samples:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def extract_bmz_profile(self, image):
        gray = image.mean(dim=0)
        H = gray.shape[0]

        top = gray[:H//3, :].mean()
        middle = gray[H//3:2*H//3, :].mean()
        bottom = gray[2*H//3:, :].mean()

        return torch.tensor([top, middle, bottom], dtype=torch.float32)

    # 🔥 BMZ MASK (unchanged but stable)
    def extract_bmz_mask(self, image):
        gray = image.mean(dim=0).cpu().numpy()

        grad = np.abs(np.gradient(gray, axis=0))
        grad = cv2.GaussianBlur(grad, (9, 9), 0)

        bmz_line = np.argmax(grad, axis=0)

        mask = np.zeros_like(gray)

        for x in range(len(bmz_line)):
            y = bmz_line[x]

            y1 = max(0, y-4)
            y2 = min(mask.shape[0], y+4)

            mask[y1:y2, x] = 1

        mask = torch.tensor(mask, dtype=torch.float32)
        return mask.unsqueeze(0)

    # 🔥 NEW: DISTANCE MAP (VERY IMPORTANT FOR NEXT STEP)
    def compute_distance_map(self, bmz_mask):
        mask = bmz_mask.squeeze(0).cpu().numpy()

    # Distance above BMZ
        dist_out = cv2.distanceTransform((1 - mask).astype(np.uint8), cv2.DIST_L2, 5)

    # Distance inside BMZ region
        dist_in = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5)

        signed_dist = dist_out - dist_in

    # Normalize to [-1, 1]
        signed_dist = signed_dist / (np.max(np.abs(signed_dist)) + 1e-6)

        return torch.tensor(signed_dist, dtype=torch.float32).unsqueeze(0)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        profile = self.extract_bmz_profile(image)
        bmz_mask = self.extract_bmz_mask(image)

        # 🔥 ADD THIS (not used yet, but ready)
        distance_map = self.compute_distance_map(bmz_mask)

        return {
            "image": image,
            "bmz_mask": bmz_mask,
            "distance_map": distance_map,   # 🔥 NEW
            "profile": profile,
            "label": torch.tensor(label, dtype=torch.long)
        }

In [79]:
dataset = AIBDDataset(DATA_DIR, transform=None)  # 🔥 NO transform here

indices = list(range(len(dataset)))
labels = [dataset.samples[i][1] for i in indices]

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# 🔥 Apply transforms AFTER split
train_dataset = Subset(AIBDDataset(DATA_DIR, transform=train_transform), train_idx)
val_dataset = Subset(AIBDDataset(DATA_DIR, transform=val_transform), val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train:", len(train_dataset), "Val:", len(val_dataset))

Total samples: 6453
Total samples: 6453
Total samples: 6453
Train: 5162 Val: 1291


In [80]:
class BMZAttention(nn.Module):
    def __init__(self, in_channels=512):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, 256, 1),
            nn.ReLU(),
            nn.Conv2d(256, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attn = self.attention(x)

        # 🔥 STABILITY: reduce attention strength slightly
        out = x * (1 + 0.8 * attn)

        return out, attn

In [81]:
class ImageEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # 🔥 CHANGE: 3 → 4 input channels
        backbone.conv1 = nn.Conv2d(
            4, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.attn = BMZAttention(512)

        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, out_dim)
        )

    def forward(self, x, bmz_mask, distance_map):
        # 🔥 ADD distance channel
        x = torch.cat([x, distance_map], dim=1)

        feat_map = self.features(x)

        bmz_mask = nn.functional.interpolate(
            bmz_mask,
            size=feat_map.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        feat_map = feat_map * (1 + 0.8 * bmz_mask)

        feat_map, attn = self.attn(feat_map)

        pooled = self.pool(feat_map).view(x.size(0), -1)
        out = self.projection(pooled)

        return out, attn

In [82]:
class BMZProfileEncoder(nn.Module):
    def __init__(self, input_dim=3, out_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),   # 🔥 ADD THIS (stabilizes learning)
            nn.Linear(32, out_dim)
        )

    def forward(self, x):
        return self.net(x)

In [83]:
class PrototypeLayer(nn.Module):
    def __init__(self, num_classes=5, feat_dim=320):
        super().__init__()
        self.prototypes = nn.Parameter(torch.randn(num_classes, feat_dim))

        self.scale = nn.Parameter(torch.tensor(10.0))  # 🔥 ADD THIS

    def forward(self, features):
        features = nn.functional.normalize(features, dim=1)
        prototypes = nn.functional.normalize(self.prototypes, dim=1)

        logits = torch.matmul(features, prototypes.T)

        # 🔥 SCALE logits (very important for focal loss + stability)
        return logits * self.scale

In [84]:
class BCPLModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.image_encoder = ImageEncoder(256)
        self.profile_encoder = BMZProfileEncoder(3, 64)

        self.prototype_layer = PrototypeLayer(len(CLASSES), 320)

    def forward(self, image, bmz_mask, distance_map, profile):
        f_img, attn = self.image_encoder(image, bmz_mask, distance_map)
        f_prof = self.profile_encoder(profile)

        # normalization (keep this)
        f_img = nn.functional.normalize(f_img, dim=1)
        f_prof = nn.functional.normalize(f_prof, dim=1)

        fused = torch.cat([f_img, f_prof], dim=1)

        logits = self.prototype_layer(fused)

        return logits, fused, attn

In [ ]:
labels_all = [dataset.samples[i][1] for i in range(len(dataset))]
class_counts = np.bincount(labels_all, minlength=len(CLASSES))

# 🔥 BASE WEIGHTS
weights = class_counts.max() / (class_counts + 1e-6)

# 🔥 FINAL FIX: Boost PF slightly
weights[CLASS_TO_IDX["PF"]] *= 1.5

weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", weights)


# 🔥 FOCAL LOSS (unchanged)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()


ce_loss_fn = FocalLoss(alpha=weights, gamma=2)


def prototype_loss(features, labels, prototypes):
    proto = prototypes[labels]
    return ((features - proto) ** 2).sum(dim=1).mean()


def attention_loss(attn):
    sparsity = attn.mean()
    sharpness = (attn * (1 - attn)).mean()
    return sparsity + sharpness


def compute_loss(logits, features, labels, model, attn):
    ce = ce_loss_fn(logits, labels)

    proto = prototype_loss(
        features,
        labels,
        model.prototype_layer.prototypes
    )

    attn_l = attention_loss(attn)

    total = ce + 0.5 * proto + 0.3 * attn_l

    return total, ce, proto, attn_l

Class counts: [2719  164 2937  197  436]
Class weights: tensor([ 1.0802, 17.9085,  1.0000, 14.9086,  6.7362])


In [86]:
model = BCPLModel().to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

best_val_acc = 0
patience = 6
counter = 0


def evaluate(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)
            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    return accuracy_score(labels_all, preds_all)


from sklearn.metrics import classification_report

def evaluate_detailed(loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            bmz_mask = batch["bmz_mask"].to(DEVICE)
            distance_map = batch["distance_map"].to(DEVICE)
            profile = batch["profile"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits, _, _ = model(images, bmz_mask, distance_map, profile)
            preds = torch.argmax(logits, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    print("\n📊 Classification Report:\n")
    print(classification_report(labels_all, preds_all, target_names=CLASSES))


for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = batch["image"].to(DEVICE)
        bmz_mask = batch["bmz_mask"].to(DEVICE)
        distance_map = batch["distance_map"].to(DEVICE)
        profile = batch["profile"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        logits, features, attn = model(images, bmz_mask, distance_map, profile)

        loss, ce, proto, attn_l = compute_loss(
            logits, features, labels, model, attn
        )

        loss.backward()

        # 🔥 Stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    train_acc = evaluate(train_loader)
    val_acc = evaluate(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss/len(train_loader):.4f}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # 🔥 Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model_bmz_distance.pth")
        print("✅ Saved Best Model")
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("⛔ Early stopping triggered")
        break


# 🔥 FINAL REPORT
evaluate_detailed(val_loader)

Epoch 1/20: 100%|██████████| 323/323 [04:30<00:00,  1.20it/s]



Epoch 1
Loss: 151.3655
Train Acc: 0.8452 | Val Acc: 0.8335
✅ Saved Best Model


Epoch 2/20: 100%|██████████| 323/323 [06:03<00:00,  1.13s/it]



Epoch 2
Loss: 142.1329
Train Acc: 0.8646 | Val Acc: 0.8451
✅ Saved Best Model


Epoch 3/20: 100%|██████████| 323/323 [06:21<00:00,  1.18s/it]



Epoch 3
Loss: 135.4526
Train Acc: 0.8611 | Val Acc: 0.8335


... (30 outputs left out)

Epoch 19/20: 100%|██████████| 323/323 [05:42<00:00,  1.06s/it]



Epoch 19
Loss: 62.2164
Train Acc: 0.9678 | Val Acc: 0.8838


Epoch 20/20: 100%|██████████| 323/323 [07:16<00:00,  1.35s/it]



Epoch 20
Loss: 59.3360
Train Acc: 0.9649 | Val Acc: 0.8730
⛔ Early stopping triggered

📊 Classification Report:

              precision    recall  f1-score   support

          PV       0.91      0.89      0.90       544
          PF       0.31      0.33      0.32        33
          BP       0.91      0.92      0.91       588
         EBA       0.41      0.51      0.45        39
  Vasculitis       0.89      0.83      0.86        87

    accuracy                           0.87      1291
   macro avg       0.68      0.70      0.69      1291
weighted avg       0.88      0.87      0.88      1291

